## MOv — Análise de Preferências Turisticas

Projeto final | SCTEC — Desenvolvimento de IA para Análise Preditiva
Abordagem: aprendizado não supervisionado (K-Means) e API FastAPI
Aplicação proposta: estudo exploratório para futura plataforma MOv em Navegantes/SC.
A base Travel Reviews não contém avaliações específicas de Navegantes. O agrupamento é um protótipo metodológico e não um recomendador turístico validado.

Sumário
1. Apresentação e objetivos
2. Ambiente e dados
3. Exploração e preparação
4. Modelagem e avaliação
5. Pipeline e persistência
6. Interpretação dos clusters
7. API FastAPI
8. Testes e documentação OpenAPI
9. Documentação e publicação
10. Conclusões e limitações

Fase 1 — Apresentação e metodologia

Problema: explorar padrões de avaliações de categorias turísticas para identificar grupos de preferências.

Objetivo geral: desenvolver e documentar um protótipo reproduzível de agrupamento e disponibilizá-lo via API.

Metodologia: base pública Travel Reviews (UCI), análise exploratória, divisão 60/20/20, padronização, seleção de K por silhouette, avaliação em teste, interpretação e integração com FastAPI.

Fonte: https://archive.ics.uci.edu/dataset/484/travel+reviews

Fase 2 — Preparação do ambiente

Etapa 2.1 — Dependências
Instalar os pacotes necessários para análise, modelagem, API e testes.

In [38]:
%pip install -q pandas numpy scikit-learn joblib fastapi uvicorn httpx pytest

Etapa 2.2 — Diretórios e CSV

O projeto usa um único diretório. O arquivo CSV pode ser enviado à raiz de /content ou já estar em data/.

In [39]:
from pathlib import Path
import shutil

BASE_DIR = Path("/content/MOv_Analise_Preditiva_SCTEC")
PASTA_DADOS = BASE_DIR / "data"
PASTA_MODELOS = BASE_DIR / "models"
PASTA_RELATORIOS = BASE_DIR / "reports"

for pasta in [PASTA_DADOS, PASTA_MODELOS, PASTA_RELATORIOS]:
    pasta.mkdir(parents=True, exist_ok=True)

CAMINHO_CSV = PASTA_DADOS / "tripadvisor_review.csv"

possiveis_origens = [
    CAMINHO_CSV,
    BASE_DIR / "tripadvisor_review.csv",
    Path("/content/tripadvisor_review.csv"),
    Path("/content/MOv/data/tripadvisor_review.csv")
]

origem = next(
    (p for p in possiveis_origens if p.is_file()),
    None
)

if origem is None:
    raise FileNotFoundError(
        "CSV não encontrado. Envie tripadvisor_review.csv "
        "para o Colab e execute esta célula novamente."
    )

if origem != CAMINHO_CSV:
    shutil.copy2(origem, CAMINHO_CSV)

print("Estrutura preparada com sucesso!")
print("CSV:", CAMINHO_CSV)
print("Tamanho:", CAMINHO_CSV.stat().st_size, "bytes")

Estrutura preparada com sucesso!
CSV: /content/MOv_Analise_Preditiva_SCTEC/data/tripadvisor_review.csv
Tamanho: 57307 bytes


Fase 3 — Exploração e preparação dos dados

Etapa 3.1 — Leitura
Inspecionar dimensões e primeiras linhas para verificar a estrutura.

In [40]:
import pandas as pd
import numpy as np

df = pd.read_csv(CAMINHO_CSV)

print("Base carregada com sucesso!")
print("Dimensões:", df.shape)

display(df.head())

Base carregada com sucesso!
Dimensões: (980, 11)


,User ID,Category 1,Category 2,Category 3,Category 4,Category 5,Category 6,Category 7,Category 8,Category 9,Category 10
0,User 1,0.93,1.8,2.29,0.62,0.80,2.42,3.19,2.79,1.82,2.42
1,User 2,1.02,2.2,2.66,0.64,1.42,3.18,3.21,2.63,1.86,2.32
2,User 3,1.22,0.8,0.54,0.53,0.24,1.54,3.18,2.80,1.31,2.50
3,User 4,0.45,1.8,0.29,0.57,0.46,1.52,3.18,2.96,1.57,2.86
4,User 5,0.51,1.2,1.18,0.57,1.54,2.02,3.18,2.78,1.18,2.54


Etapa 3.2 — Qualidade dos dados

Verificar tipos, ausências, duplicatas e estatísticas descritivas. Duplicatas devem ser analisadas antes de qualquer remoção.

In [41]:
print("Colunas:")
print(df.columns.tolist())

print("\nTipos de dados:")
display(df.dtypes)

print("\nValores ausentes:")
display(df.isnull().sum())

print("\nLinhas duplicadas:", df.duplicated().sum())

print("\nResumo estatístico:")
display(df.iloc[:, 1:].describe())

Colunas:
['User ID', 'Category 1', 'Category 2', 'Category 3', 'Category 4', 'Category 5', 'Category 6', 'Category 7', 'Category 8', 'Category 9', 'Category 10']

Tipos de dados:


,0
User ID,object
Category 1,float64
Category 2,float64
Category 3,float64
Category 4,float64
Category 5,float64
Category 6,float64
Category 7,float64
Category 8,float64
Category 9,float64



Valores ausentes:


,0
User ID,0
Category 1,0
Category 2,0
Category 3,0
Category 4,0
Category 5,0
Category 6,0
Category 7,0
Category 8,0
Category 9,0



Linhas duplicadas: 0

Resumo estatístico:


,Category 1,Category 2,Category 3,Category 4,Category 5,Category 6,Category 7,Category 8,Category 9,Category 10
count,980.000000,980.000000,980.000000,980.000000,980.000000,980.000000,980.000000,980.000000,980.000000,980.000000
mean,0.893194,1.352612,1.013306,0.532500,0.939735,1.842898,3.180939,2.835061,1.569439,2.799224
std,0.326912,0.478280,0.788607,0.279731,0.437430,0.539538,0.007824,0.137505,0.364629,0.321380
min,0.340000,0.000000,0.130000,0.150000,0.060000,0.140000,3.160000,2.420000,0.740000,2.140000
25%,0.670000,1.080000,0.270000,0.410000,0.640000,1.460000,3.180000,2.740000,1.310000,2.540000
50%,0.830000,1.280000,0.820000,0.500000,0.900000,1.800000,3.180000,2.820000,1.540000,2.780000
75%,1.020000,1.560000,1.572500,0.580000,1.200000,2.200000,3.180000,2.910000,1.760000,3.040000
max,3.220000,3.640000,3.620000,3.440000,3.300000,3.760000,3.210000,3.390000,3.170000,3.660000


Etapa 3.3 — Dispersão das categorias

Comparar mínimos, máximos, médias, desvios e amplitudes para justificar a padronização.

In [42]:
variacao = df.iloc[:, 1:].agg(
    ["min", "max", "mean", "std"]
).T

variacao["amplitude"] = (
    variacao["max"] - variacao["min"]
)

display(variacao.round(4))

,min,max,mean,std,amplitude
Category 1,0.34,3.22,0.8932,0.3269,2.88
Category 2,0.00,3.64,1.3526,0.4783,3.64
Category 3,0.13,3.62,1.0133,0.7886,3.49
Category 4,0.15,3.44,0.5325,0.2797,3.29
Category 5,0.06,3.30,0.9397,0.4374,3.24
Category 6,0.14,3.76,1.8429,0.5395,3.62
Category 7,3.16,3.21,3.1809,0.0078,0.05
Category 8,2.42,3.39,2.8351,0.1375,0.97
Category 9,0.74,3.17,1.5694,0.3646,2.43
Category 10,2.14,3.66,2.7992,0.3214,1.52


Etapa 3.4 — Nomes descritivos

Renomear as dez categorias conforme a descrição do conjunto de dados. Confirmar o mapeamento com a fonte antes da entrega.

In [43]:
novos_nomes = {
    "Category 1": "galerias_arte",
    "Category 2": "clubes_danca",
    "Category 3": "bares_sucos",
    "Category 4": "restaurantes",
    "Category 5": "museus",
    "Category 6": "resorts",
    "Category 7": "parques_picnic",
    "Category 8": "praias",
    "Category 9": "teatros",
    "Category 10": "instituicoes_religiosas"
}

df = df.rename(columns=novos_nomes)

COLUNAS_MODELO = list(novos_nomes.values())

print("Categorias organizadas:")
print(df.columns.tolist())

Categorias organizadas:
['User ID', 'galerias_arte', 'clubes_danca', 'bares_sucos', 'restaurantes', 'museus', 'resorts', 'parques_picnic', 'praias', 'teatros', 'instituicoes_religiosas']


Etapa 3.5 — Seleção das variáveis

Excluir User ID do modelo, pois identifica registros e não expressa preferência.

In [44]:
usuarios = df["User ID"].copy()

X = df[COLUNAS_MODELO].copy()

if X.isnull().any().any():
    raise ValueError("Existem valores ausentes nas variáveis.")

if not all(pd.api.types.is_numeric_dtype(X[c]) for c in X):
    raise TypeError("Existem variáveis não numéricas.")

print("Dimensões das preferências:", X.shape)
print("Quantidade de variáveis:", X.shape[1])

Dimensões das preferências: (980, 10)
Quantidade de variáveis: 10


Etapa 3.6 — Divisão treino/validação/teste

Separar 60%, 20% e 20%, com semente fixa, para evitar usar os dados de teste na escolha do número de grupos.

In [45]:
from sklearn.model_selection import train_test_split

X_temp, X_teste = train_test_split(
    X,
    test_size=0.20,
    random_state=42
)

X_treino, X_validacao = train_test_split(
    X_temp,
    test_size=0.25,
    random_state=42
)

print("Treino:", X_treino.shape)
print("Validação:", X_validacao.shape)
print("Teste:", X_teste.shape)

Treino: (588, 10)
Validação: (196, 10)
Teste: (196, 10)


Etapa 3.7 — Padronização

Ajustar StandardScaler apenas no treino e aplicar a transformação à validação e ao teste, prevenindo vazamento de dados.

In [46]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_treino_scaled = scaler.fit_transform(X_treino)
X_validacao_scaled = scaler.transform(X_validacao)
X_teste_scaled = scaler.transform(X_teste)

print("Treino padronizado:", X_treino_scaled.shape)
print("Validação padronizada:", X_validacao_scaled.shape)
print("Teste padronizado:", X_teste_scaled.shape)

print("\nMédias do treino:")
print(X_treino_scaled.mean(axis=0).round(4))

print("\nDesvios-padrão do treino:")
print(X_treino_scaled.std(axis=0).round(4))

Treino padronizado: (588, 10)
Validação padronizada: (196, 10)
Teste padronizado: (196, 10)

Médias do treino:
[-0.  0. -0.  0. -0. -0.  0.  0.  0.  0.]

Desvios-padrão do treino:
[1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]


Fase 4 — Modelagem e avaliação

Etapa 4.1 — Seleção de K

Treinar K-Means para K de 2 a 8 e comparar a silhouette na validação. Silhouette é medida de separação/cohesão, não acurácia de classificação.

In [47]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

resultados_k = []

for k in range(2, 9):
    modelo = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    modelo.fit(X_treino_scaled)

    rotulos = modelo.predict(X_validacao_scaled)

    if len(np.unique(rotulos)) < 2:
        silhouette = np.nan
    else:
        silhouette = silhouette_score(
            X_validacao_scaled,
            rotulos
        )

    resultados_k.append({
        "k": k,
        "silhouette_validacao": silhouette
    })

resultados_k_df = pd.DataFrame(resultados_k)

display(resultados_k_df.round(4))

,k,silhouette_validacao
0,2,0.1866
1,3,0.1365
2,4,0.1367
3,5,0.1321
4,6,0.1315
5,7,0.1232
6,8,0.1175


Etapa 4.2 — Escolha automática

Escolher o maior valor válido de silhouette na validação, sem fixar o resultado histórico manualmente.

In [48]:
if resultados_k_df["silhouette_validacao"].isna().all():
    raise ValueError("Nenhum K gerou grupos válidos.")

melhor_linha = resultados_k_df.loc[
    resultados_k_df["silhouette_validacao"].idxmax()
]

k_final = int(melhor_linha["k"])
silhouette_validacao = float(
    melhor_linha["silhouette_validacao"]
)

print("K escolhido:", k_final)
print(
    "Silhouette na validação:",
    round(silhouette_validacao, 4)
)

K escolhido: 2
Silhouette na validação: 0.1866


Etapa 4.3 — Avaliação no teste

Após escolher K, avaliar uma única vez no conjunto de teste. Resultados numéricos devem ser lidos da execução atual.

In [49]:
modelo_final = KMeans(
    n_clusters=k_final,
    random_state=42,
    n_init=10
)

modelo_final.fit(X_treino_scaled)

rotulos_teste = modelo_final.predict(X_teste_scaled)

if len(np.unique(rotulos_teste)) < 2:
    silhouette_teste = np.nan
else:
    silhouette_teste = silhouette_score(
        X_teste_scaled,
        rotulos_teste
    )

print("Número de grupos:", k_final)
print("Silhouette de validação:", round(silhouette_validacao, 4))
print("Silhouette de teste:", round(silhouette_teste, 4))

Número de grupos: 2
Silhouette de validação: 0.1866
Silhouette de teste: 0.2185


Fase 5 — Pipeline e persistência

Etapa 5.1 — Pipeline

Unir StandardScaler e KMeans em um objeto único. O pipeline usa a mesma divisão de treino e os mesmos parâmetros da avaliação.

In [50]:
from sklearn.pipeline import Pipeline

pipeline_mov = Pipeline([
    ("scaler", StandardScaler()),
    ("kmeans", KMeans(
        n_clusters=k_final,
        random_state=42,
        n_init=10
    ))
])

pipeline_mov.fit(X_treino)

print("Pipeline treinado!")
print("Etapas:", list(pipeline_mov.named_steps.keys()))
print("Variáveis:", pipeline_mov.n_features_in_)

Pipeline treinado!
Etapas: ['scaler', 'kmeans']
Variáveis: 10


Etapa 5.2 — Serialização com Joblib

Salvar o pipeline treinado em models/, para reutilização na API.

In [51]:
import joblib

CAMINHO_MODELO = (
    PASTA_MODELOS / "mov_kmeans_pipeline.joblib"
)

joblib.dump(pipeline_mov, CAMINHO_MODELO)

print("Modelo salvo:", CAMINHO_MODELO)
print("Existe:", CAMINHO_MODELO.is_file())
print("Tamanho:", CAMINHO_MODELO.stat().st_size, "bytes")

Modelo salvo: /content/MOv_Analise_Preditiva_SCTEC/models/mov_kmeans_pipeline.joblib
Existe: True
Tamanho: 4345 bytes


Etapa 5.3 — Teste de recarga

Confirmar que o objeto recarregado produz os mesmos grupos no conjunto de teste.

In [52]:
modelo_carregado = joblib.load(CAMINHO_MODELO)

previsoes_original = pipeline_mov.predict(X_teste)
previsoes_carregado = modelo_carregado.predict(X_teste)

assert np.array_equal(
    previsoes_original,
    previsoes_carregado
), "As previsões não coincidem."

print("Modelo carregado com sucesso!")
print("Previsões idênticas: True")

Modelo carregado com sucesso!
Previsões idênticas: True


Fase 6 — Interpretação dos grupos

Etapa 6.1 — Perfis dos clusters

Usar as médias originais de cada categoria e a quantidade de registros por grupo. Os números dos grupos não indicam qualidade nem hierarquia.

In [53]:
import pandas as pd
import joblib
from pathlib import Path

# Localizar os arquivos do projeto
BASE_DIR = Path("/content/MOv_Analise_Preditiva_SCTEC")

CAMINHO_CSV = BASE_DIR / "data" / "tripadvisor_review.csv"
CAMINHO_MODELO = BASE_DIR / "models" / "mov_kmeans_pipeline.joblib"

# Carregar os dados e o modelo salvo
df_perfis = pd.read_csv(CAMINHO_CSV)
modelo_carregado = joblib.load(CAMINHO_MODELO)

# Renomear as categorias turísticas
novos_nomes = {
    "Category 1": "galerias_arte",
    "Category 2": "clubes_danca",
    "Category 3": "bares_sucos",
    "Category 4": "restaurantes",
    "Category 5": "museus",
    "Category 6": "resorts",
    "Category 7": "parques_picnic",
    "Category 8": "praias",
    "Category 9": "teatros",
    "Category 10": "instituicoes_religiosas"
}

df_perfis = df_perfis.rename(columns=novos_nomes)

# Recuperar a ordem das variáveis do modelo
colunas_modelo = list(modelo_carregado.feature_names_in_)

# Preparar os dados para inferência
X_perfis = df_perfis[colunas_modelo].copy()

# Identificar o cluster de cada registro
df_perfis["grupo"] = modelo_carregado.predict(X_perfis)

# Contar a quantidade de registros por grupo
quantidade_grupos = (
    df_perfis["grupo"]
    .value_counts()
    .sort_index()
)

# Calcular a média das categorias em cada grupo
perfil_grupos = (
    df_perfis.groupby("grupo")[colunas_modelo]
    .mean()
    .round(3)
)

print("MOv — ANÁLISE DOS CLUSTERS")
print("-" * 45)

print("\n1. Quantidade de registros por grupo:")
display(quantidade_grupos.to_frame("quantidade"))

print("\n2. Média das categorias por grupo:")
display(perfil_grupos.T)

print("\n3. Total de registros analisados:")
print(len(df_perfis))

MOv — ANÁLISE DOS CLUSTERS
---------------------------------------------

1. Quantidade de registros por grupo:


,quantidade
grupo,
0,404
1,576



2. Média das categorias por grupo:


grupo,0,1
galerias_arte,0.900,0.888
clubes_danca,1.470,1.270
bares_sucos,1.609,0.596
restaurantes,0.644,0.454
museus,1.221,0.742
resorts,2.247,1.559
parques_picnic,3.187,3.177
praias,2.813,2.850
teatros,1.582,1.561
instituicoes_religiosas,2.554,2.971



3. Total de registros analisados:
980


Etapa 6.2 — Registro de métricas

Salvar as métricas calculadas nesta execução, evitando constantes fixas que possam ficar inconsistentes com o modelo.

In [54]:
import json
from pathlib import Path

PASTA_RELATORIOS = BASE_DIR / "reports"
PASTA_RELATORIOS.mkdir(parents=True, exist_ok=True)

metricas = {
    "projeto": "MOv - Analise de Preferencias Turisticas",
    "algoritmo": "KMeans",
    "tipo_aprendizado": "Nao supervisionado",
    "total_registros": int(len(df_perfis)),
    "numero_variaveis": int(len(colunas_modelo)),
    "numero_clusters": int(k_final),
    "silhouette_validacao": float(silhouette_validacao),
    "silhouette_teste": float(silhouette_teste),
    "distribuicao_clusters": {
        str(grupo): int(quantidade)
        for grupo, quantidade in quantidade_grupos.items()
    }
}

CAMINHO_METRICAS = (
    PASTA_RELATORIOS / "metricas_modelo.json"
)

with open(CAMINHO_METRICAS, "w", encoding="utf-8") as arquivo:
    json.dump(metricas, arquivo, indent=4, ensure_ascii=False)

print("Relatório de métricas salvo!")
print("Caminho:", CAMINHO_METRICAS)
print("Arquivo encontrado:", CAMINHO_METRICAS.exists())

print("\nConteúdo do relatório:")

Relatório de métricas salvo!
Caminho: /content/MOv_Analise_Preditiva_SCTEC/reports/metricas_modelo.json
Arquivo encontrado: True

Conteúdo do relatório:


Fase 7 — Desenvolvimento da API FastAPI

A arquitetura separa schemas (validação), services (inferência), routers (HTTP) e main (inicialização e carregamento único do modelo).

Etapa 7.1 — schemas.py

In [55]:
codigo_schemas = '''
from pydantic import BaseModel, Field, ConfigDict


class PreferenciasTurista(BaseModel):
    model_config = ConfigDict(extra="forbid")

    galerias_arte: float = Field(ge=0, le=4)
    clubes_danca: float = Field(ge=0, le=4)
    bares_sucos: float = Field(ge=0, le=4)
    restaurantes: float = Field(ge=0, le=4)
    museus: float = Field(ge=0, le=4)
    resorts: float = Field(ge=0, le=4)
    parques_picnic: float = Field(ge=0, le=4)
    praias: float = Field(ge=0, le=4)
    teatros: float = Field(ge=0, le=4)
    instituicoes_religiosas: float = Field(ge=0, le=4)


class ResultadoPredicao(BaseModel):
    grupo: int
    mensagem: str
'''

(BASE_DIR / "schemas.py").write_text(
    codigo_schemas.strip() + "\n",
    encoding="utf-8"
)

print("schemas.py criado!")

schemas.py criado!


Etapa 7.2 — services.py

In [56]:
codigo_services = '''
import pandas as pd

NOMES_VARIAVEIS = [
    "galerias_arte",
    "clubes_danca",
    "bares_sucos",
    "restaurantes",
    "museus",
    "resorts",
    "parques_picnic",
    "praias",
    "teatros",
    "instituicoes_religiosas"
]


def prever_grupo(modelo, preferencias):
    dados = preferencias.model_dump()

    entrada = pd.DataFrame(
        [dados],
        columns=NOMES_VARIAVEIS
    )

    grupo = int(modelo.predict(entrada)[0])

    return {
        "grupo": grupo,
        "mensagem": (
            f"Preferências classificadas no grupo {grupo}."
        )
    }
'''

(BASE_DIR / "services.py").write_text(
    codigo_services.strip() + "\n",
    encoding="utf-8"
)

print("services.py criado!")

services.py criado!


Etapa 7.3 — routers.py

In [57]:
codigo_routers = '''
from fastapi import APIRouter, Request, HTTPException
from schemas import PreferenciasTurista, ResultadoPredicao
from services import prever_grupo

router = APIRouter()

@router.get("/health")
def verificar_saude(request: Request):
    modelo = getattr(request.app.state, "modelo", None)

    return {
        "status": "ativo" if modelo is not None else "indisponivel",
        "modelo_carregado": modelo is not None
    }

@router.post("/predict", response_model=ResultadoPredicao)
def realizar_predicao(
    preferencias: PreferenciasTurista,
    request: Request
):
    modelo = getattr(request.app.state, "modelo", None)

    if modelo is None:
        raise HTTPException(
            status_code=503,
            detail="Modelo indisponível."
        )

    return prever_grupo(modelo, preferencias)
'''

(BASE_DIR / "routers.py").write_text(
    codigo_routers.strip() + "\n",
    encoding="utf-8"
)

print("routers.py criado!")

routers.py criado!


Etapa 7.4 — main.py

In [58]:
codigo_main = '''
from contextlib import asynccontextmanager
from pathlib import Path

import joblib
from fastapi import FastAPI
from routers import router

BASE_DIR = Path(__file__).resolve().parent
CAMINHO_MODELO = BASE_DIR / "models" / "mov_kmeans_pipeline.joblib"

@asynccontextmanager
async def lifespan(app: FastAPI):
    app.state.modelo = joblib.load(CAMINHO_MODELO)
    try:
        yield
    finally:
        app.state.modelo = None

app = FastAPI(
    title="MOv - Análise de Preferências Turísticas",
    description="API para agrupamento de perfis turísticos.",
    version="1.0.0",
    lifespan=lifespan
)

app.include_router(router)
'''

(BASE_DIR / "main.py").write_text(
    codigo_main.strip() + "\n",
    encoding="utf-8"
)

print("main.py criado!")

main.py criado!


Fase 8 — Testes e documentação da API

Etapa 8.1 — Teste funcional

Testar GET /health e POST /predict com o modelo carregado no ciclo de vida da aplicação.

In [59]:
import sys
import importlib
from fastapi.testclient import TestClient

sys.path.insert(0, str(BASE_DIR))
importlib.invalidate_caches()

for nome in ["main", "routers", "services", "schemas"]:
    sys.modules.pop(nome, None)

from main import app

preferencias_teste = {
    "galerias_arte": 3.0,
    "clubes_danca": 1.0,
    "bares_sucos": 2.0,
    "restaurantes": 4.0,
    "museus": 3.0,
    "resorts": 2.0,
    "parques_picnic": 4.0,
    "praias": 4.0,
    "teatros": 2.0,
    "instituicoes_religiosas": 1.0
}

with TestClient(app) as cliente:
    health = cliente.get("/health")
    predict = cliente.post(
        "/predict",
        json=preferencias_teste
    )

    print("HEALTH:", health.status_code, health.json())
    print("PREDICT:", predict.status_code, predict.json())

    assert health.status_code == 200
    assert health.json()["modelo_carregado"] is True
    assert predict.status_code == 200
    assert isinstance(predict.json()["grupo"], int)

print("Testes aprovados!")

HEALTH: 200 {'status': 'ativo', 'modelo_carregado': True}
PREDICT: 200 {'grupo': 0, 'mensagem': 'Preferências classificadas no grupo 0.'}
Testes aprovados!


Etapa 8.2 — Entrada inválida

Confirmar que uma preferência fora do intervalo é rejeitada com HTTP 422.

In [60]:
dados_invalidos = preferencias_teste.copy()
dados_invalidos["praias"] = -1

with TestClient(app) as cliente:
    resposta = cliente.post(
        "/predict",
        json=dados_invalidos
    )

assert resposta.status_code == 422

print("Validação de entrada aprovada!")

Validação de entrada aprovada!


Etapa 8.3 — Testes automatizados

Criar testes reproduzíveis para saúde, predição e validação de dados.

In [61]:
from pathlib import Path

BASE_DIR = Path("/content/MOv_Analise_Preditiva_SCTEC")

codigo_testes = '''
from fastapi.testclient import TestClient
from main import app

EXEMPLO = {
    "galerias_arte": 3.0,
    "clubes_danca": 1.0,
    "bares_sucos": 2.0,
    "restaurantes": 4.0,
    "museus": 3.0,
    "resorts": 2.0,
    "parques_picnic": 4.0,
    "praias": 4.0,
    "teatros": 2.0,
    "instituicoes_religiosas": 1.0
}

def test_health():
    with TestClient(app) as cliente:
        resposta = cliente.get("/health")
        assert resposta.status_code == 200
        assert resposta.json()["modelo_carregado"] is True

def test_predict():
    with TestClient(app) as cliente:
        resposta = cliente.post("/predict", json=EXEMPLO)
        assert resposta.status_code == 200
        assert isinstance(resposta.json()["grupo"], int)

def test_valor_invalido():
    dados = EXEMPLO.copy()
    dados["praias"] = -1

    with TestClient(app) as cliente:
        resposta = cliente.post("/predict", json=dados)
        assert resposta.status_code == 422

def test_campo_extra():
    dados = EXEMPLO.copy()
    dados["categoria_inexistente"] = 2.0

    with TestClient(app) as cliente:
        resposta = cliente.post("/predict", json=dados)
        assert resposta.status_code == 422
'''

caminho_testes = BASE_DIR / "test_api.py"
caminho_testes.write_text(codigo_testes, encoding="utf-8")

print("Arquivo de testes criado!")
print("Caminho:", caminho_testes)
print("Arquivo encontrado:", caminho_testes.exists())

Arquivo de testes criado!
Caminho: /content/MOv_Analise_Preditiva_SCTEC/test_api.py
Arquivo encontrado: True


Etapa 8.4 — Execução do Pytest

O código de saída deve ser 0. Uma mensagem anterior de aprovação não substitui esta execução.

In [62]:
import subprocess
import sys

resultado = subprocess.run(
    [
        sys.executable,
        "-m",
        "pytest",
        "test_api.py",
        "-v"
    ],
    cwd=str(BASE_DIR),
    capture_output=True,
    text=True
)

print("RESULTADO DOS TESTES AUTOMATIZADOS")
print("-" * 45)
print(resultado.stdout)

if resultado.stderr:
    print("MENSAGENS ADICIONAIS:")
    print(resultado.stderr)

print("Código de saída:", resultado.returncode)
assert resultado.returncode == 0, "Falha nos testes automatizados"

RESULTADO DOS TESTES AUTOMATIZADOS
---------------------------------------------
============================= test session starts ==============================
platform linux -- Python 3.13.18, pytest-8.4.2, pluggy-1.7.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/MOv_Analise_Preditiva_SCTEC
plugins: platformdirs-4.13.5, anyio-4.15.1, typeguard-4.6.0, langsmith-0.14.12
collecting ... collected 4 items

test_api.py::test_health PASSED                                          [ 25%]
test_api.py::test_predict PASSED                                         [ 50%]
test_api.py::test_valor_invalido PASSED                                  [ 75%]
test_api.py::test_campo_extra PASSED                                     [100%]

============================== 4 passed in 2.05s ===============================

Código de saída: 0


Etapa 8.5 — Auditoria HTTP

Verificar códigos 404, 422, 503 e 500; o último é uma falha simulada, não um comportamento desejado.

In [63]:
from fastapi.testclient import TestClient
from main import app

print("MOv — AUDITORIA DE RESPOSTAS HTTP")
print("-" * 45)

with TestClient(app) as cliente:

    # Teste 1: campo obrigatório ausente
    dados_incompletos = preferencias_teste.copy()
    dados_incompletos.pop("praias")

    resposta = cliente.post(
        "/predict",
        json=dados_incompletos
    )
    print("Campo obrigatório ausente:", resposta.status_code)

    # Teste 2: valor acima do limite
    dados_fora_limite = preferencias_teste.copy()
    dados_fora_limite["praias"] = 5.0

    resposta = cliente.post(
        "/predict",
        json=dados_fora_limite
    )
    print("Valor acima do limite:", resposta.status_code)

    # Teste 3: texto não numérico
    dados_texto = preferencias_teste.copy()
    dados_texto["praias"] = "invalido"

    resposta = cliente.post(
        "/predict",
        json=dados_texto
    )
    print("Texto não numérico:", resposta.status_code)

    # Teste 4: rota inexistente
    resposta = cliente.get("/rota_inexistente")
    print("Rota inexistente:", resposta.status_code)

    # Teste 5: documentação Swagger
    resposta = cliente.get("/docs")
    print("Swagger /docs:", resposta.status_code)

print("-" * 45)
print("Auditoria inicial concluída.")

MOv — AUDITORIA DE RESPOSTAS HTTP
---------------------------------------------
Campo obrigatório ausente: 422
Valor acima do limite: 422
Texto não numérico: 422
Rota inexistente: 404
Swagger /docs: 200
---------------------------------------------
Auditoria inicial concluída.


In [64]:
from fastapi.testclient import TestClient
from main import app

print("MOv — TESTE HTTP 503")
print("-" * 40)

with TestClient(app) as cliente:

    # Guardar o modelo original
    modelo_original = app.state.modelo

    try:
        # Simular indisponibilidade temporária
        app.state.modelo = None

        resposta = cliente.post(
            "/predict",
            json=preferencias_teste
        )

        print("Código HTTP:", resposta.status_code)
        print("Resposta:", resposta.json())

        assert resposta.status_code == 503

        print("\nTESTE APROVADO!")

    finally:
        # Restaurar o modelo original
        app.state.modelo = modelo_original

print("Modelo restaurado após o teste.")

MOv — TESTE HTTP 503
----------------------------------------
Código HTTP: 503
Resposta: {'detail': 'Modelo indisponível.'}

TESTE APROVADO!
Modelo restaurado após o teste.


In [65]:
from unittest.mock import patch
from fastapi.testclient import TestClient
from main import app

print("MOv — TESTE DE FALHA INTERNA")
print("-" * 45)

with TestClient(app, raise_server_exceptions=False) as cliente:

    with patch(
        "routers.prever_grupo",
        side_effect=RuntimeError("Falha simulada na inferência")
    ):
        resposta = cliente.post(
            "/predict",
            json=preferencias_teste
        )

    print("Código HTTP:", resposta.status_code)
    print("Resposta:", resposta.text)

    assert resposta.status_code == 500

print("\nTeste de erro interno concluído.")

MOv — TESTE DE FALHA INTERNA
---------------------------------------------
Código HTTP: 500
Resposta: Internal Server Error

Teste de erro interno concluído.


Etapa 8.6 — Documentação OpenAPI

Confirmar as rotas e os esquemas publicados. A interface Swagger pode ser acessada em /docs quando o servidor estiver em execução.

In [66]:
from fastapi.testclient import TestClient
from main import app

print("MOv — DOCUMENTAÇÃO SWAGGER / OPENAPI")
print("-" * 45)

with TestClient(app) as cliente:
    resposta = cliente.get("/openapi.json")
    documentacao = resposta.json()

    print("HTTP OpenAPI:", resposta.status_code)
    print("Título:", documentacao["info"]["title"])
    print("Versão:", documentacao["info"]["version"])

    print("\nENDPOINTS DOCUMENTADOS:")

    for caminho, metodos in documentacao["paths"].items():
        for metodo in metodos:
            print(f"{metodo.upper()} {caminho}")

    print("\nESQUEMAS DISPONÍVEIS:")

    for esquema in documentacao["components"]["schemas"]:
        print("-", esquema)

    assert "/health" in documentacao["paths"]
    assert "/predict" in documentacao["paths"]
    assert "post" in documentacao["paths"]["/predict"]

print("\nDocumentação OpenAPI validada!")

MOv — DOCUMENTAÇÃO SWAGGER / OPENAPI
---------------------------------------------
HTTP OpenAPI: 200
Título: MOv - Análise de Preferências Turísticas
Versão: 1.0.0

ENDPOINTS DOCUMENTADOS:
GET /health
POST /predict

ESQUEMAS DISPONÍVEIS:
- HTTPValidationError
- PreferenciasTurista
- ResultadoPredicao
- ValidationError

Documentação OpenAPI validada!


Fase 9 — Documentação e entrega

Etapa 9.1 — requirements.txt

Registrar as versões instaladas para facilitar a reprodução.

In [67]:
from importlib.metadata import version
from pathlib import Path

BASE_DIR = Path("/content/MOv_Analise_Preditiva_SCTEC")

dependencias = [
    "fastapi",
    "uvicorn",
    "pydantic",
    "pandas",
    "numpy",
    "scikit-learn",
    "joblib",
    "pytest",
    "httpx"
]

linhas = [
    f"{pacote}=={version(pacote)}"
    for pacote in dependencias
]

caminho_requirements = BASE_DIR / "requirements.txt"

caminho_requirements.write_text(
    "\n".join(linhas) + "\n",
    encoding="utf-8"
)

print("REQUIREMENTS.TXT CRIADO")
print("-" * 40)
print(caminho_requirements.read_text(encoding="utf-8"))
print("Arquivo encontrado:", caminho_requirements.exists())

REQUIREMENTS.TXT CRIADO
----------------------------------------
fastapi==0.141.1
uvicorn==0.54.0
pydantic==2.14.1
pandas==2.2.3
numpy==2.1.3
scikit-learn==1.6.1
joblib==1.6.0
pytest==8.4.2
httpx==0.28.1

Arquivo encontrado: True


Etapa 9.2 — README.md

Documentar instalação, execução, endpoints, métricas, limitações e repositório. Os valores abaixo são gerados a partir das métricas desta sessão.

In [68]:
from pathlib import Path

BASE_DIR = Path("/content/MOv_Analise_Preditiva_SCTEC")

readme = """# MOv — Análise de Preferências Turísticas

Projeto Final — Desenvolvimento de IA para Análise Preditiva | SCTEC

## 1. Objetivo

Desenvolver um protótipo de Machine Learning capaz de
identificar agrupamentos de preferências turísticas,
como estudo inicial para a futura plataforma MOv,
voltada à personalização de experiências em
Navegantes, Santa Catarina.

## 2. Base de dados

- Arquivo: `data/tripadvisor_review.csv`
- Registros: 980
- Variáveis de preferência: 10
- Identificador adicional: User ID
- Referência pública: Travel Reviews — UCI Machine Learning Repository
- Link: https://archive.ics.uci.edu/dataset/484/travel+reviews

A base não representa especificamente visitantes de Navegantes.
O projeto tem caráter experimental.

## 3. Metodologia

- Análise exploratória dos dados
- Divisão: 60% treinamento, 20% validação, 20% teste
- Padronização com StandardScaler
- Agrupamento não supervisionado com K-Means
- Avaliação de K entre 2 e 8
- Seleção de 2 clusters
- Serialização com Joblib

O modelo salvo utiliza um Pipeline com StandardScaler e KMeans.

## 4. Resultados

| Métrica | Resultado |
|---|---|
| Silhouette de validação | VAL_SIL |
| Silhouette de teste | TEST_SIL |
| Número de clusters | K_FINAL |
| Registros no grupo 0 | GRUPO_ZERO |
| Registros no grupo 1 | GRUPO_UM |

Os resultados indicam separação relativamente fraca entre
os agrupamentos. A segmentação deve ser interpretada
como um protótipo exploratório, não como uma
recomendação turística já validada.

Métricas detalhadas: `reports/metricas_modelo.json`.

## 5. Estrutura do projeto

```text
data/
    tripadvisor_review.csv
models/
    mov_kmeans_pipeline.joblib
reports/
    metricas_modelo.json
main.py
routers.py
schemas.py
services.py
test_api.py
requirements.txt
README.md
```

O notebook de treinamento também deverá ser incluído
no repositório final.

## 6. Tecnologias

- Python
- Pandas e NumPy
- Scikit-learn
- Joblib
- FastAPI
- Pydantic
- Uvicorn
- Pytest

## 7. Instalação

Requisitos: Python e Git instalados.

Clone o repositório e entre na pasta do projeto.

Crie um ambiente virtual:

```bash
python -m venv .venv
```

Ative o ambiente virtual.

Windows:

```bash
.venv\\Scripts\\activate
```

Linux/macOS:

```bash
source .venv/bin/activate
```

Instale as dependências:

```bash
python -m pip install -r requirements.txt
```

## 8. Executar a API

Na pasta principal do projeto:

```bash
uvicorn main:app --reload
```

Acesse:

- Swagger: http://127.0.0.1:8000/docs
- Health: http://127.0.0.1:8000/health
- OpenAPI: http://127.0.0.1:8000/openapi.json

## 9. Endpoints

### GET /health

Verifica se o modelo está carregado na memória.

Resposta de exemplo:

```json
{
  "status": "ativo",
  "modelo_carregado": true
}
```

### POST /predict

Recebe dez valores de preferências turísticas
e retorna o grupo atribuído pelo modelo.

Exemplo de requisição:

```json
{
  "galerias_arte": 3.0,
  "clubes_danca": 1.0,
  "bares_sucos": 2.0,
  "restaurantes": 4.0,
  "museus": 3.0,
  "resorts": 2.0,
  "parques_picnic": 4.0,
  "praias": 4.0,
  "teatros": 2.0,
  "instituicoes_religiosas": 1.0
}
```

Resposta observada:

```json
{
  "grupo": 0,
  "mensagem": "Preferências classificadas no grupo 0."
}
```

## 10. Validação e testes

A API utiliza Pydantic para validar os dados de entrada.

Códigos HTTP verificados:

- 200: operação bem-sucedida
- 404: rota inexistente
- 422: entrada inválida
- 503: modelo indisponível
- 500: erro interno inesperado

Para executar os testes automatizados:

```bash
python -m pytest -v
```

Foram aprovados quatro testes automatizados.

## 11. Limitações e melhorias futuras

- A base não representa diretamente o turismo de Navegantes.
- A separação entre clusters é relativamente fraca.
- O modelo identifica agrupamentos, não prevê destinos
  nem garante recomendações personalizadas.
- Coletar dados próprios de visitantes.
- Investigar novas variáveis e algoritmos.
- Relacionar perfis a atrações e experiências locais.
- Avaliar a utilidade das recomendações com usuários reais.

## 12. Vídeo de demonstração

Link: a inserir após a gravação e publicação.

## 13. Desenvolvimento

Projeto acadêmico desenvolvido no âmbito do SCTEC.
"""

readme = (readme.replace("VAL_SIL", f"{silhouette_validacao:.4f}")
    .replace("TEST_SIL", f"{silhouette_teste:.4f}")
    .replace("K_FINAL", str(k_final))
    .replace("GRUPO_ZERO", str(quantidade_grupos.get(0, 0)))
    .replace("GRUPO_UM", str(quantidade_grupos.get(1, 0))))

caminho_readme = BASE_DIR / "README.md"
caminho_readme.write_text(readme, encoding="utf-8")

print("README CRIADO COM SUCESSO!")
print("Caminho:", caminho_readme)
print("Arquivo encontrado:", caminho_readme.exists())
print("Quantidade de caracteres:", len(readme))

README CRIADO COM SUCESSO!
Caminho: /content/MOv_Analise_Preditiva_SCTEC/README.md
Arquivo encontrado: True
Quantidade de caracteres: 4156


Etapa 9.3 — Conferência dos arquivos

Verificar se os módulos, o modelo, os relatórios e o README estão presentes.

In [69]:
from pathlib import Path

BASE_DIR = Path("/content/MOv_Analise_Preditiva_SCTEC")

print("MOv — CONFERÊNCIA PARA O GITHUB")
print("-" * 45)

# Verificar o README
readme = (BASE_DIR / "README.md").read_text(
    encoding="utf-8"
)

print("README encontrado: True")
print("Títulos Markdown:", readme.count("\n## "))
print("Blocos de código:", readme.count("```"))

if readme.count("```") % 2 == 0:
    print("Blocos de código: delimitadores equilibrados")
else:
    print("ATENÇÃO: delimitadores de código desequilibrados")

# Procurar notebooks na pasta do projeto e no Colab
notebooks = list(BASE_DIR.rglob("*.ipynb"))
notebooks += [
    p for p in Path("/content").glob("*.ipynb")
    if p not in notebooks
]

print("\nNOTEBOOKS ENCONTRADOS:")
if notebooks:
    for notebook in notebooks:
        print("-", notebook)
else:
    print("Nenhum arquivo .ipynb encontrado nessas pastas.")

print("\nARQUIVOS PRINCIPAIS:")
for nome in [
    "main.py",
    "routers.py",
    "schemas.py",
    "services.py",
    "test_api.py",
    "requirements.txt",
    "README.md",
    "data/tripadvisor_review.csv",
    "models/mov_kmeans_pipeline.joblib",
    "reports/metricas_modelo.json"
]:
    print(
        nome,
        "OK" if (BASE_DIR / nome).is_file() else "AUSENTE"
    )

MOv — CONFERÊNCIA PARA O GITHUB
---------------------------------------------
README encontrado: True
Títulos Markdown: 13
Blocos de código: 20
Blocos de código: delimitadores equilibrados

NOTEBOOKS ENCONTRADOS:
Nenhum arquivo .ipynb encontrado nessas pastas.

ARQUIVOS PRINCIPAIS:
main.py OK
routers.py OK
schemas.py OK
services.py OK
test_api.py OK
requirements.txt OK
README.md OK
data/tripadvisor_review.csv OK
models/mov_kmeans_pipeline.joblib OK
reports/metricas_modelo.json OK


Etapa 9.4 — Backup do projeto

Criar um ZIP dos arquivos gerados. O notebook .ipynb deve ser salvo separadamente no Colab e incluído no GitHub.

In [70]:
from pathlib import Path
import zipfile

BASE_DIR = Path("/content/MOv_Analise_Preditiva_SCTEC")
CAMINHO_ZIP = Path("/content/MOv_Backup_Projeto_SCTEC.zip")

with zipfile.ZipFile(
    CAMINHO_ZIP,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as zipf:

    for arquivo in BASE_DIR.rglob("*"):
        if not arquivo.is_file():
            continue

        if any(
            pasta in {"__pycache__", ".pytest_cache", ".git"}
            for pasta in arquivo.relative_to(BASE_DIR).parts
        ):
            continue

        zipf.write(
            arquivo,
            arcname=arquivo.relative_to(BASE_DIR.parent)
        )

print("BACKUP DO PROJETO MOv")
print("-" * 45)
print("Arquivo criado:", CAMINHO_ZIP.exists())
print("Nome:", CAMINHO_ZIP.name)
print("Tamanho:", round(CAMINHO_ZIP.stat().st_size / 1024, 2), "KB")

BACKUP DO PROJETO MOv
---------------------------------------------
Arquivo criado: True
Nome: MOv_Backup_Projeto_SCTEC.zip
Tamanho: 22.27 KB


Etapa 9.5 — Publicação
1. Salvar o notebook no Colab e baixar uma cópia .ipynb.
2. Publicar no GitHub o notebook, módulos Python, CSV (se a licença permitir), modelo, métricas, requirements.txt e README.
3. Conferir os links e executar os testes após a publicação.
4. Documentar branches/commits (GitFlow) e gravar o vídeo, caso exigidos no enunciado.
5. Enviar os links solicitados no AVA antes do prazo.


Fase 10 — Conclusões e limitações
Célula 70 de 70

Fase 10 — Conclusões e limitações
O MOv demonstra um fluxo de análise exploratória, segmentação não supervisionada e disponibilização por API. A silhouette relativamente baixa registrada historicamente sugere grupos pouco separados; a interpretação definitiva deve considerar as métricas da execução atual.

Limitações: dados não locais; clusters não equivalem a recomendações validadas; rótulos dos clusters são arbitrários; resultados dependem da base e das variáveis.

Próximos passos: coletar dados de visitantes de Navegantes, testar outros métodos, validar perfis com usuários e associar grupos a experiências turísticas reais.

Conferindo métricas

In [71]:
from pathlib import Path
import json

# Procurar o relatório nas duas estruturas usadas anteriormente
caminhos = [
    Path("/content/MOv/reports/metricas_modelo.json"),
    Path("/content/MOv_Analise_Preditiva_SCTEC/reports/metricas_modelo.json")
]

encontrados = [p for p in caminhos if p.is_file()]

if not encontrados:
    print("RELATÓRIO AINDA NÃO ENCONTRADO.")
    print("Será necessário gerar o JSON após treinar o modelo.")
else:
    for caminho in encontrados:
        print("\nRELATÓRIO ENCONTRADO:", caminho)

        with open(caminho, "r", encoding="utf-8") as arquivo:
            metricas = json.load(arquivo)

        print("\nMÉTRICAS REGISTRADAS:")
        print(json.dumps(metricas, indent=4, ensure_ascii=False))


RELATÓRIO ENCONTRADO: /content/MOv_Analise_Preditiva_SCTEC/reports/metricas_modelo.json

MÉTRICAS REGISTRADAS:
{
    "projeto": "MOv - Analise de Preferencias Turisticas",
    "algoritmo": "KMeans",
    "tipo_aprendizado": "Nao supervisionado",
    "total_registros": 980,
    "numero_variaveis": 10,
    "numero_clusters": 2,
    "silhouette_validacao": 0.18663282604405232,
    "silhouette_teste": 0.21853858768873685,
    "distribuicao_clusters": {
        "0": 404,
        "1": 576
    }
}


Conferência do modelo salvo

In [72]:
import joblib
from pathlib import Path

caminho_modelo = Path(
    "/content/MOv_Analise_Preditiva_SCTEC/models/mov_kmeans_pipeline.joblib"
)

if caminho_modelo.is_file():
    modelo = joblib.load(caminho_modelo)

    print("Modelo encontrado: True")
    print("Tipo:", type(modelo).__name__)
    print("Etapas:", list(modelo.named_steps.keys()))
    print("Variáveis:", modelo.n_features_in_)
    print("Clusters:", modelo.named_steps["kmeans"].n_clusters)
else:
    print("Modelo não encontrado.")

Modelo encontrado: True
Tipo: Pipeline
Etapas: ['scaler', 'kmeans']
Variáveis: 10
Clusters: 2


Testes de API

In [73]:
import subprocess

resultado = subprocess.run(
    [
        "python", "-m", "pytest",
        "/content/MOv_Analise_Preditiva_SCTEC/test_api.py",
        "-v"
    ],
    capture_output=True,
    text=True
)

print(resultado.stdout)
print(resultado.stderr)
print("Código de saída:", resultado.returncode)

============================= test session starts ==============================
platform linux -- Python 3.13.18, pytest-8.4.2, pluggy-1.7.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content
plugins: platformdirs-4.13.5, anyio-4.15.1, typeguard-4.6.0, langsmith-0.14.12
collecting ... collected 4 items

MOv_Analise_Preditiva_SCTEC/test_api.py::test_health PASSED              [ 25%]
MOv_Analise_Preditiva_SCTEC/test_api.py::test_predict PASSED             [ 50%]
MOv_Analise_Preditiva_SCTEC/test_api.py::test_valor_invalido PASSED      [ 75%]
MOv_Analise_Preditiva_SCTEC/test_api.py::test_campo_extra PASSED         [100%]

============================== 4 passed in 1.78s ===============================


Código de saída: 0


Conferência de Swagger

In [74]:
from fastapi.testclient import TestClient
from main import app

with TestClient(app) as cliente:
    resposta = cliente.get("/openapi.json")

    print("Status OpenAPI:", resposta.status_code)

    if resposta.status_code == 200:
        esquema = resposta.json()
        print("Título:", esquema["info"]["title"])
        print("Rotas documentadas:", list(esquema["paths"].keys()))

Status OpenAPI: 200
Título: MOv - Análise de Preferências Turísticas
Rotas documentadas: ['/health', '/predict']


Conferência de arquivos para o GitHub

In [75]:
from pathlib import Path

pasta = Path("/content/MOv_Analise_Preditiva_SCTEC")

arquivos = [
    "main.py",
    "routers.py",
    "schemas.py",
    "services.py",
    "test_api.py",
    "requirements.txt",
    "README.md",
    "data/tripadvisor_review.csv",
    "models/mov_kmeans_pipeline.joblib",
    "reports/metricas_modelo.json"
]

print("CONFERÊNCIA DOS ARQUIVOS DO MOv")
print("-" * 40)

for arquivo in arquivos:
    status = "OK" if (pasta / arquivo).is_file() else "PENDENTE"
    print(f"{status}: {arquivo}")

CONFERÊNCIA DOS ARQUIVOS DO MOv
----------------------------------------
OK: main.py
OK: routers.py
OK: schemas.py
OK: services.py
OK: test_api.py
OK: requirements.txt
OK: README.md
OK: data/tripadvisor_review.csv
OK: models/mov_kmeans_pipeline.joblib
OK: reports/metricas_modelo.json


Conferir o notebook

In [76]:
from pathlib import Path

pasta = Path("/content/MOv_Analise_Preditiva_SCTEC")

notebooks = list(pasta.rglob("*.ipynb"))

print("NOTEBOOKS ENCONTRADOS:")
if notebooks:
    for arquivo in notebooks:
        print(arquivo.relative_to(pasta))
else:
    print("Nenhum notebook dentro da pasta do projeto.")

NOTEBOOKS ENCONTRADOS:
Nenhum notebook dentro da pasta do projeto.


In [77]:
from pathlib import Path
import shutil

pasta_projeto = Path("/content/MOv_Analise_Preditiva_SCTEC")
arquivo_zip = Path("/content/MOv_Projeto_GitHub.zip")

shutil.make_archive(
    str(arquivo_zip.with_suffix("")),
    "zip",
    root_dir=pasta_projeto
)

print("Arquivo ZIP criado:", arquivo_zip.exists())
print("Local:", arquivo_zip)

Arquivo ZIP criado: True
Local: /content/MOv_Projeto_GitHub.zip
